# NF-CSE-CIC-IDS2018-v3 — Phase 3: data preprocessing (cleaning)

**Owner:** Ahmed Hamouda. Applies the approved decisions of phase 2 for this dataset
(`report/data_problems/phase2_data_problems.pdf`, section 2 and section 6). Phase 2 evidence: `02_cic18.ipynb`.
The result has exactly the same 56 columns as `UNSW_clean.parquet`.

| Step | What | Decision |
|---|---|---|
| 0 | Load the raw CSV | – |
| 1 | Drop the two `..._SECOND_BYTES` columns | C3 |
| 2 | `L7_PROTO` → `L7_MAIN` + `L7_SUB` (= 0, no decimals in this file) | C3 |
| 3 | Remove exact duplicate rows (captures recorded twice) | C4 |
| 4 | Remove **all** copies of flows that carry two different labels | C5 |
| 5 | Remove DoS SlowHTTPTest (refused FTP connections) and the port-21 flows labelled SSH brute force | C6 |
| 6 | Remove attack flows whose behaviour also appears as Benign | C7 |
| 7 | One row per (behaviour, label), with a `count` column | Section 6 |
| 8 | Shared `clean()`: drop the IP columns, NaN / inf, duplicates | M3 + shared rule |
| 9 | `map_labels()` (11 labels added to the taxonomy, C1) and drop `Label` | C1, C2 |
| 10 | Checks and save | – |

**Not done here** (on the merged data, identical for all four datasets): M1, M6, M7, M8, M9. **Kept on purpose:** small
classes (C8), FTP-BruteForce as a single refused-connection pattern (C10), port 0, ICMPv6, LLMNR, congestion flags,
oversized packets, TTL = 0, outliers.

20 million rows: about 10 GB of memory; close other heavy programs before running.

## Step 0 — Load the raw data

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv

from src.config import *

pd.set_option("display.max_columns", 100)
SOURCE = "CIC18"
path = RAW_DIR / DATASETS[SOURCE]["file"]

header = pd.read_csv(path, nrows=0).columns
# int32 halves the memory; pyarrow stops with an error if a value does not fit. L7_PROTO is read as an integer:
# this would fail if CIC-2018 had decimal sub-codes like UNSW, so loading also proves it has none.
types = {c: pa.int32() for c in header if c not in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"]}
for c in ["FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "DNS_TTL_ANSWER"]:
    types[c] = pa.int64()
table = pv.read_csv(path, convert_options=pv.ConvertOptions(column_types=types))
df = table.to_pandas(self_destruct=True)
del table

raw_labels = df["Attack"].value_counts()
steps = []
def log(step, decision, what):
    steps.append({"step": step, "decision": decision, "rows": len(df), "columns": df.shape[1], "what": what})
    print(f"{step}: {len(df):,} rows x {df.shape[1]} columns  ({what})")

def removed(mask, name):
    """Show what a removal takes away, per label."""
    print(f"{name}: {int(mask.sum()):,} rows")
    display(df.loc[mask, "Attack"].value_counts().rename("removed").to_frame()
            .join(df["Attack"].value_counts().rename("before")).assign(after=lambda t: t["before"] - t["removed"]))

log("0 load", "-", "raw file")
raw_labels

0 load: 20,115,529 rows x 55 columns  (raw file)


Attack
Benign                      17514626
DDOS_attack-HOIC             1032311
FTP-BruteForce                386720
DDoS_attacks-LOIC-HTTP        288589
Bot                           207703
SSH-Bruteforce                188474
Infilteration                 188152
DoS_attacks-SlowHTTPTest      105550
DoS_attacks-Hulk              100076
DoS_attacks-GoldenEye          61300
DoS_attacks-Slowloris          36040
DDOS_attack-LOIC-UDP            3450
Brute_Force_-Web                1618
Brute_Force_-XSS                 480
SQL_Injection                    440
Name: count, dtype: int64

## Step 1 — Drop the two `..._SECOND_BYTES` columns (C3)
Here they are integers without NaN, but they carry the same wrong meaning as in UNSW (bytes per millisecond, directions
swapped) and are dropped in every dataset, so the four cleaned files have the same columns.

In [2]:
df = df.drop(columns=["SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES"])
log("1 drop SECOND_BYTES", "C3", "2 columns removed")

1 drop SECOND_BYTES: 20,115,529 rows x 53 columns  (2 columns removed)


## Step 2 — `L7_PROTO` → `L7_MAIN` + `L7_SUB` (C3)
CIC-2018 has no sub-codes (the column loaded as an integer), so `L7_MAIN` = `L7_PROTO` and `L7_SUB` = 0: the same two
columns as the cleaned UNSW file.

In [3]:
pos = df.columns.get_loc("L7_PROTO")
df.insert(pos, "L7_MAIN", df["L7_PROTO"].astype("int32"))
df.insert(pos + 1, "L7_SUB", np.zeros(len(df), dtype="int32"))
df = df.drop(columns="L7_PROTO")
log("2 split L7_PROTO", "C3", "L7_PROTO -> L7_MAIN + L7_SUB = 0")

2 split L7_PROTO: 20,115,529 rows x 54 columns  (L7_PROTO -> L7_MAIN + L7_SUB = 0)


## Step 3 — Remove exact duplicate rows (C4)
Rows identical in every column (including IPs and the millisecond timestamp): whole attack captures were written twice.
Compared through a 64-bit hash of each row (same result as `df.duplicated()`, far less memory on 20 million rows).

In [4]:
row_hash = pd.util.hash_pandas_object(df, index=False).to_numpy()
dup = pd.Series(row_hash).duplicated().to_numpy()
removed(dup, "exact duplicates")
df = df[~dup]
del row_hash
log("3 exact duplicates", "C4", f"{int(dup.sum()):,} copies removed")

exact duplicates: 628,474 rows


,removed,before,after
Attack,,,
FTP-BruteForce,193360,386720,193360
Benign,121872,17514626,17392754
Bot,96243,207703,111460
SSH-Bruteforce,94237,188474,94237
Infilteration,72347,188152,115805
DoS_attacks-GoldenEye,30650,61300,30650
DoS_attacks-Slowloris,18020,36040,18020
DDOS_attack-LOIC-UDP,1725,3450,1725
Brute_Force_-XSS,20,480,460


3 exact duplicates: 19,487,055 rows x 54 columns  (628,474 copies removed)


## Step 4 — Remove flows that carry two labels (C5)
The same flow (identical in every column except the labels, same IPs and millisecond) is labelled XSS in one row and SQL
injection or web brute force in another. The correct label cannot be known, so **all** copies go.

In [5]:
feat = [c for c in df.columns if c not in ("Label", "Attack")]
fh = pd.util.hash_pandas_object(df[feat], index=False).to_numpy()
n_labels = pd.Series(df["Attack"].to_numpy()).groupby(fh).nunique()
two = np.isin(fh, n_labels.index[n_labels > 1])
removed(two, "flows with two labels")
df = df[~two]
del fh
log("4 two labels", "C5", f"{int(two.sum()):,} rows removed")

flows with two labels: 880 rows


,removed,before,after
Attack,,,
Brute_Force_-XSS,440,460,20
SQL_Injection,272,440,168
Brute_Force_-Web,168,1618,1450


4 two labels: 19,486,175 rows x 54 columns  (880 rows removed)


## Step 5 — Remove the mislabelled SlowHTTPTest flows (C6)
Every DoS SlowHTTPTest flow is a refused FTP connection (port 21, one SYN, a reset), identical to FTP-BruteForce: no HTTP
at all. The few port-21 flows labelled SSH brute force are the same pattern.

In [6]:
bad = ((df["Attack"] == "DoS_attacks-SlowHTTPTest") |
       ((df["Attack"] == "SSH-Bruteforce") & (df["L4_DST_PORT"] == 21))).to_numpy()
removed(bad, "SlowHTTPTest + port-21 SSH")
df = df[~bad]
log("5 SlowHTTPTest", "C6", f"{int(bad.sum()):,} rows removed")

SlowHTTPTest + port-21 SSH: 105,580 rows


,removed,before,after
Attack,,,
DoS_attacks-SlowHTTPTest,105550,105550,0
SSH-Bruteforce,30,94237,94207


5 SlowHTTPTest: 19,380,595 rows x 54 columns  (105,580 rows removed)


## Step 6 — Remove attack flows that look exactly like Benign flows (C7)
In CIC-2018 everything an infected machine did during an attack window was labelled as the attack, including its normal
DNS lookups (phase 2: the same machine sent the identical flow labelled Benign on other days). An attack flow whose
behaviour (`behaviour_hash`, the same definition as step 7) also appears as Benign in this dataset is removed. This
rule is **for CIC-2018 only** (ToN-IoT and BoT-IoT keep theirs: T6, B7).

In [7]:
h = behaviour_hash(df)
is_benign = (df["Attack"] == "Benign").to_numpy()
same = ~is_benign & np.isin(h, np.unique(h[is_benign]))
removed(same, "attack flows identical to Benign")
df = df[~same]
del h
log("6 attack = Benign behaviour", "C7", f"{int(same.sum()):,} rows removed")

attack flows identical to Benign: 95,558 rows


,removed,before,after
Attack,,,
Infilteration,86596,115805,29209
DDoS_attacks-LOIC-HTTP,8325,288589,280264
Brute_Force_-Web,502,1450,948
SQL_Injection,114,168,54
Brute_Force_-XSS,20,20,0
DoS_attacks-Hulk,1,100076,100075


6 attack = Benign behaviour: 19,285,037 rows x 54 columns  (95,558 rows removed)


## Step 7 — One row per (behaviour, label), with `count` (section 6)
Repeats of the same behaviour with the same label become one row; `count` keeps how many flows it stands for. Runs before
the IPs are dropped, so different machines doing the same thing are counted as separate flows.

In [8]:
flows_before = len(df)
df = collapse_behaviours(df, "Attack")
print(f"{flows_before:,} flows -> {len(df):,} rows; sum of count = {df['count'].sum():,}")
assert df["count"].sum() == flows_before
display(pd.DataFrame({"raw flows": raw_labels, "rows kept": df["Attack"].value_counts(),
                      "largest count": df.groupby("Attack")["count"].max()}).fillna(0).astype(int).sort_values("raw flows", ascending=False))
log("7 collapse behaviours", "section 6", "repeats -> one row + count")

19,285,037 flows -> 8,945,792 rows; sum of count = 19,285,037


,raw flows,rows kept,largest count
Attack,,,
Benign,17514626,8178232,90163
DDOS_attack-HOIC,1032311,244060,3826
FTP-BruteForce,386720,1,193360
DDoS_attacks-LOIC-HTTP,288589,265826,37
Bot,207703,2804,14689
SSH-Bruteforce,188474,93979,3
Infilteration,188152,27213,46
DoS_attacks-SlowHTTPTest,105550,0,0
DoS_attacks-Hulk,100076,100071,4


7 collapse behaviours: 8,945,792 rows x 55 columns  (repeats -> one row + count)


## Step 8 — Shared `clean()` (`src/config.py`)

In [9]:
rows_before = len(df)
df, stats = clean(df, SOURCE)
print("rows lost in clean():", rows_before - len(df))
log("8 clean()", "M3 + shared rule", f"IP columns dropped; {rows_before - len(df)} rows lost")
stats

rows lost in clean(): 0
8 clean(): 8,945,792 rows x 53 columns  (IP columns dropped; 0 rows lost)


{'source_dataset': 'CIC18',
 'rows_raw': 8945792,
 'cols_raw': 55,
 'id_columns_dropped': 'IPV4_SRC_ADDR, IPV4_DST_ADDR',
 'inf_values': 0,
 'nan_values': 0,
 'imputed_columns': 'none',
 'rows_with_nan': 0,
 'duplicate_rows': 0,
 'rows_clean': 8945792,
 'cols_clean': 53}

## Step 9 — Shared labels (`map_labels`) and drop `Label`
The 11 CIC-2018 labels missing from the taxonomy were added to `src/config.py` (decision C1): clean names that keep the
attack tool (e.g. `DDOS_attack-HOIC` → "DDoS HOIC", `Infilteration` → "Infiltration").

In [10]:
df = map_labels(df, "Attack", SOURCE)
assert (df["is_attack"].astype(int) == df["Label"].astype(int)).all(), "Label and is_attack disagree"
df = df.drop(columns="Label")
display(df.groupby(["attack_family", "attack_label"], observed=True)["count"].agg(rows="size", flows="sum"))
log("9 map_labels + drop Label", "C1, C2", "4 label columns added, Label dropped")

rows     flows
attack_family              attack_label                      
Benign                     Benign           8178232  17392754
Credential attacks         FTP brute force        1    193360
                           SSH brute force    93979     94207
                           Web brute force      910       948
Denial of service          DDoS HOIC         244060   1032311
                           DDoS LOIC-HTTP    265826    280264
                           DDoS LOIC-UDP       1723      1725
                           DoS GoldenEye      27410     30650
                           DoS Hulk          100071    100075
                           DoS Slowloris       3510     18020
Exploitation and intrusion Infiltration       27213     29209
Malware and botnets        Bot                 2804    111460
Web and injection          SQL injection         53        54

9 map_labels + drop Label: 8,945,792 rows x 56 columns  (4 label columns added, Label dropped)


## Step 10 — Checks and save

In [11]:
checks = validate(df)
print("validate(): total violations", int(checks["violations"].sum()))
num = df.select_dtypes("number")
print("NaN cells:", int(df.isna().sum().sum()), "| inf cells:", int(np.isinf(num).sum().sum()))
print("IP columns left:", [c for c in df.columns if "ADDR" in c])
import pyarrow.parquet as pq
unsw_file = PROCESSED_DIR / "UNSW_clean.parquet"
unsw_cols = pq.read_schema(unsw_file).names if unsw_file.exists() else None
print("same columns and order as UNSW_clean:", unsw_cols == list(df.columns) if unsw_cols else "UNSW file not found")
print(f"final: {len(df):,} rows x {df.shape[1]} columns | flows represented {int(df['count'].sum()):,} of {int(raw_labels.sum()):,} raw")

validate(): total violations 0


NaN cells: 0 | inf cells: 0
IP columns left: []
same columns and order as UNSW_clean: True
final: 8,945,792 rows x 56 columns | flows represented 19,285,037 of 20,115,529 raw


In [12]:
log("10 final", "-", "cleaned table")
lab_map = map_labels(pd.DataFrame({"Attack": raw_labels.index}), "Attack", SOURCE).set_index("Attack")["attack_label"]
labels = cleaning_report(steps, raw_labels.groupby(lab_map).sum(), df, SOURCE)
out = save_clean(df, SOURCE)
print("saved", out.relative_to(ROOT), f"({out.stat().st_size / 1e6:.0f} MB)")
display(pd.DataFrame(steps))
labels.sort_values("flows_raw", ascending=False)

10 final: 8,945,792 rows x 56 columns  (cleaned table)


saved data\processed\CIC18_clean.parquet (518 MB)


,step,decision,rows,columns,what
0,0 load,-,20115529,55,raw file
1,1 drop SECOND_BYTES,C3,20115529,53,2 columns removed
2,2 split L7_PROTO,C3,20115529,54,L7_PROTO -> L7_MAIN + L7_SUB = 0
3,3 exact duplicates,C4,19487055,54,"628,474 copies removed"
4,4 two labels,C5,19486175,54,880 rows removed
5,5 SlowHTTPTest,C6,19380595,54,"105,580 rows removed"
6,6 attack = Benign behaviour,C7,19285037,54,"95,558 rows removed"
7,7 collapse behaviours,section 6,8945792,55,repeats -> one row + count
8,8 clean(),M3 + shared rule,8945792,53,IP columns dropped; 0 rows lost
9,9 map_labels + drop Label,"C1, C2",8945792,56,"4 label columns added, Label dropped"


,source_dataset,flows_raw,rows_clean,flows_represented,attack_family
attack_label,,,,,
Benign,CIC18,17514626,8178232,17392754,Benign
DDoS HOIC,CIC18,1032311,244060,1032311,Denial of service
FTP brute force,CIC18,386720,1,193360,Credential attacks
DDoS LOIC-HTTP,CIC18,288589,265826,280264,Denial of service
Bot,CIC18,207703,2804,111460,Malware and botnets
SSH brute force,CIC18,188474,93979,94207,Credential attacks
Infiltration,CIC18,188152,27213,29209,Exploitation and intrusion
DoS SlowHTTPTest,CIC18,105550,0,0,Denial of service
DoS Hulk,CIC18,100076,100071,100075,Denial of service
